# Classical sentiment baselines

## Objective and methodology

Compare real TF-IDF Logistic Regression and Linear SVM runs. Vocabulary/IDF fitting uses training only; C/class weights use validation only. Reload a trained checkpoint and check its prediction schema without retraining from a notebook.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/"configs/phase3_config.yaml").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(ROOT/"scripts"))
result = json.loads((ROOT/"reports/phase3_results.json").read_text())
assert result["mode"] == "full", "A smoke run is not a completed experiment"
print("Dataset sizes:", {k:v["reviews"] for k,v in result["dataset"]["splits"].items()})
print("All evaluation labels are rating-derived weak labels, not verified human sentiment.")

Dataset sizes: {'train': 18367, 'validation': 2489, 'test': 2451}
All evaluation labels are rating-derived weak labels, not verified human sentiment.


## Completed run and checkpoint validation

In [2]:
from src.models.inference import ReviewPredictor
names = [n for n in result["models"] if n in ["logistic_regression", "linear_svm"]]
display(pd.DataFrame({n:{"validation_macro_f1":result["models"][n]["validation"]["macro_f1"], "weak_test_accuracy":result["models"][n]["test"]["accuracy"], "weak_test_macro_f1":result["models"][n]["test"]["macro_f1"]} for n in names}).T)
selected = result["selection"]["best_baseline"]
predictor = ReviewPredictor(ROOT/result["models"][selected]["artifact_path"])
example = predictor.single("The sound is clear but the battery is poor", "audio_devices")
print("Synthetic inference demonstration; not a benchmark:", {k:example[k] for k in ["sentiment","confidence_semantics","model_version"]})

,validation_macro_f1,weak_test_accuracy,weak_test_macro_f1
logistic_regression,0.659306,0.846185,0.672835
linear_svm,0.648824,0.879233,0.666704


Synthetic inference demonstration; not a benchmark: {'sentiment': 'neutral', 'confidence_semantics': 'uncalibrated_probability', 'model_version': 'phase3-full-logistic_regression-seed42'}


## Actual figures

External image links keep notebook outputs small.

In [3]:
figures = [name for name in result.get("figures",[]) if ("loss" in name if "06_" in '05_sentiment_baseline' else "model_comparison" in name)]
for name in figures:
    display(Markdown("![Actual experimental figure](../reports/figures/"+name+")"))

![Actual experimental figure](../reports/figures/phase3_full_model_comparison.png)

## Interpretation and limitations

Linear baselines provide a low-cost weak-label reference. Review held-out class-wise metrics, not just accuracy; the result does not establish aspect sentiment or human-language understanding.